# SMS Guard — запуск в Google Colab

Блокнот скачивает проект, устанавливает зависимости, готовит данные, обучает baseline-модель (задания 3–7),
запускает тесты и поднимает Django-сайт со ссылкой для открытия в браузере. Выполняйте ячейки сверху вниз.

## 1. Получить проект (выполните ОДИН из вариантов)

**Вариант А — из GitHub** (репозиторий должен быть публичным):

In [ ]:
import os
if not os.path.exists("/content/project_AI"):
    !git clone https://github.com/ruslanalimov800-max/project_AI.git
%cd /content/project_AI

**Вариант Б — загрузить zip-архив проекта** (если репозиторий приватный). Папку `.git` и `.venv` в архив класть не нужно:

In [ ]:
from google.colab import files
import zipfile, glob, os

name = next(iter(files.upload()))
zipfile.ZipFile(name).extractall("/content/project")
root = os.path.dirname(glob.glob("/content/project/**/manage.py", recursive=True)[0])
os.chdir(root)
print("Готово:", os.getcwd())

## 2. Установить зависимости

pandas, NumPy и scikit-learn в Colab уже есть, ставим только недостающее.

In [ ]:
!pip install -q django djangorestframework python-dotenv pytest pytest-django

## 3. Данные и обучение (задания 3–7)

Модель обучается заново, чтобы файл `.joblib` соответствовал версии scikit-learn в Colab.

In [ ]:
!python -m ml.data.prepare_sms

In [ ]:
!python -m ml.sms_baseline

## 4. Тесты

In [ ]:
!python -m pytest -q

## 5. Метрики и графики

In [ ]:
import json
from IPython.display import Image, display

report = json.load(open("ml/artifacts/sms_baseline_report.json", encoding="utf-8"))
print("Размеры выборок:", report["sizes"])
print("Метрики на валидации:", json.dumps(report["validation"], indent=2))
for name in ["sms_baseline_confusion", "sms_classes", "sms_indicators"]:
    display(Image(f"docs/figures/{name}.png"))

## 6. Проверка модели на своих текстах

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from ml.sms_baseline import predict_sms

for text in [
    "WINNER!! You have won a 1000 prize. Call 09061701461 now to claim",
    "Ok, see you at lunch tomorrow",
    "Free entry! Text WIN to 80086 to receive your reward",
]:
    print(predict_sms(text), "<-", text)

## 7. Запустить сайт (Django)

In [ ]:
%%writefile .env
DEBUG=1
ALLOWED_HOSTS=*

In [ ]:
!python manage.py migrate
!python manage.py check

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, time, urllib.request
subprocess.Popen(["python", "manage.py", "runserver", "8000", "--noreload"],
                 stdout=open("/tmp/django.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
print("Health внутри Colab:", urllib.request.urlopen("http://127.0.0.1:8000/api/v1/health/").read().decode())

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    print(line, end="")
    if "trycloudflare.com" in line and "https" in line:
        break
print("\nОткройте ссылку выше (главная) и добавьте /api/v1/health/ для проверки API.")
print("Чтобы остановить сайт — остановите эту ячейку.")